# KiteFS Feature Store - Getting Started Showcase

This notebook demonstrates how to use KiteFS.

- **Phase 1**  Define feature groups, ingest prepared data, explore the local store, and train an initial model — entirely on your machine, no cloud setup needed. 
- **Phase 2**  Publish the feature store to AWS (S3 + DynamoDB) and materialize online features for serving. 
- **Phase 3**  One month later: ingest new data, refresh the online store, build a rolling training dataset, and retrain the model. 

**Scenario:** A Turkish real-estate platform wants to recommend a realistic sale price for new listings. The data set is containing the data of year 2025. Two feature groups capture what KiteFS stores and serves:

| Feature Group | Storage | Entity Key | Event Timestamp | Purpose |
|---|---|---|---|---|
| `listing_features` | Offline only | `listing_id` | `sold_at` | Historical sold listings used for training |
| `town_market_features` | Offline + Online | `town_id` | `event_timestamp` | Monthly town-level price aggregates for training and serving |

### About the Dataset

The `data/` directory holds synthetic, prepared feature data for a Turkish real-estate platform operating in six towns — Kadikoy, Besiktas, Tuzla (Istanbul) and Cankaya, Kecioren, Mamak (Ankara). The story spans two moments in time: a full year of **2025** sales history used to train the first price model in **January 2026**, and a fresh **January 2026** batch that arrives one month later to drive a retraining loop. KiteFS does not compute these features — a data pipeline would normally produce them; here, seeded generator scripts in `scripts/` play that role.

**2025 history (first model):**
- `listing_features.parquet` — 10,000 sold listings (Jan–Dec 2025). One row per property with `listing_id`, `town_id`, `net_area`, `number_of_rooms`, `build_year`, `sold_price`, and the `sold_at` event timestamp. The model predicts `sold_price` from these attributes.
- `town_market_features.parquet` — 72 rows: a monthly `avg_price_per_sqm` per town (6 towns × 12 months). Each aggregate's `event_timestamp` is the **first day of the following month** (e.g. January → `2025-02-01`), so the point-in-time join can never leak a market average that wasn't yet published.

**January 2026 batch (retraining):**
- `listing_features_jan2026.parquet` — 1,000 new sold listings (`listing_id` starts at 20001 to avoid colliding with 2025 IDs).
- `town_market_jan2026.parquet` — 6 rows, one `avg_price_per_sqm` per town, stamped `event_timestamp = 2026-02-01`.

### Feature Group Definitions

During this demo we will need couple of feature groups.
- `listing_features` - This feature group will be used to store the features related to individual real estate listings. It will be used for training our model.
- `town_market_features` - This feature group will be used to store aggregated market features at the town level. It will be used for both training and serving.
- `demo_validation_modes` - This will contain two feature groups to demonstrate different validation modes.

After `kitefs init`, use the feature group definitions to use them during the demo.

In [ ]:
# town_market_features.py
from kitefs import FeatureGroup, Feature, EntityKey, EventTimestamp
from kitefs import FeatureType, StorageTarget, Expect
from kitefs import ValidationMode, Metadata

town_market_features = FeatureGroup(
    name="town_market_features",
    storage_target=StorageTarget.OFFLINE_AND_ONLINE,
    entity_key=EntityKey(name="town_id", dtype=FeatureType.INTEGER),
    event_timestamp=EventTimestamp(name="event_timestamp"),
    features=[
        Feature(
            name="avg_price_per_sqm",
            dtype=FeatureType.FLOAT,
            expect=Expect().not_null().gt(0),
        )
    ],
    ingestion_validation=ValidationMode.ERROR,
    offline_retrieval_validation=ValidationMode.NONE,
    metadata=Metadata(
        description="Monthly town-level market aggregate",
        owner="data-science-team",
        tags={"domain": "real-estate", "cadence": "monthly"},
    ),
)

# listing_features.py
from kitefs import (
    EntityKey,
    EventTimestamp,
    Expect,
    Feature,
    FeatureGroup,
    FeatureType,
    JoinKey,
    Metadata,
    StorageTarget,
    ValidationMode,
)


listing_features = FeatureGroup(
    name="listing_features",
    storage_target=StorageTarget.OFFLINE,
    entity_key=EntityKey(name="listing_id", dtype=FeatureType.INTEGER),
    event_timestamp=EventTimestamp(name="sold_at"),
    features=[
        Feature(
            name="net_area", dtype=FeatureType.INTEGER, expect=Expect().not_null().gt(0)
        ),
        Feature(
            name="number_of_rooms",
            dtype=FeatureType.INTEGER,
            expect=Expect().not_null().gt(0),
        ),
        Feature(
            name="build_year",
            dtype=FeatureType.INTEGER,
            expect=Expect().not_null().gte(1900).lte(2030),
        ),
        Feature(
            name="sold_price", dtype=FeatureType.FLOAT, expect=Expect().not_null().gt(0)
        ),
    ],
    join_keys=[
        JoinKey(
            name="town_id",
            dtype=FeatureType.INTEGER,
            referenced_group="town_market_features",
        )
    ],
    ingestion_validation=ValidationMode.ERROR,
    offline_retrieval_validation=ValidationMode.NONE,
    metadata=Metadata(
        description="Historical sold listing attributes and prices",
        owner="data-science-team",
        tags={"domain": "real-estate", "cadence": "monthly"},
    ),
)


# demo_validation_modes.py
from kitefs import EntityKey, EventTimestamp, Expect, Feature, FeatureGroup
from kitefs import FeatureType, Metadata, StorageTarget, ValidationMode


demo_filter_features = FeatureGroup(
    name="demo_filter_features",
    storage_target=StorageTarget.OFFLINE,
    entity_key=EntityKey(name="demo_id", dtype=FeatureType.INTEGER),
    event_timestamp=EventTimestamp(name="event_timestamp"),
    features=[
        Feature(name="score", dtype=FeatureType.FLOAT, expect=Expect().not_null().gt(0))
    ],
    ingestion_validation=ValidationMode.FILTER,
    offline_retrieval_validation=ValidationMode.NONE,
    metadata=Metadata(
        description="Demo-only group for FILTER validation mode",
        owner="data-science-team",
        tags={"purpose": "validation-demo"},
    ),
)


demo_none_features = FeatureGroup(
    name="demo_none_features",
    storage_target=StorageTarget.OFFLINE,
    entity_key=EntityKey(name="demo_id", dtype=FeatureType.INTEGER),
    event_timestamp=EventTimestamp(name="event_timestamp"),
    features=[
        Feature(name="score", dtype=FeatureType.FLOAT, expect=Expect().not_null().gt(0))
    ],
    ingestion_validation=ValidationMode.NONE,
    offline_retrieval_validation=ValidationMode.NONE,
    metadata=Metadata(
        description="Demo-only group for NONE validation mode",
        owner="data-science-team",
        tags={"purpose": "validation-demo"},
    ),
)

## Phase 1 - Local Development & Training

Start with the local runtime target (`KITEFS_RUNTIME_TARGET=local`, which is the default). KiteFS writes everything under `feature_store/` — no AWS credentials or cloud access needed at this stage.

### 1.1 Create a local FeatureStore

`FeatureStore()` reads `./kitefs.yaml`, resolves the runtime target, and builds the storage provider for that target. The instance is bound to its target for its lifetime; switching targets requires creating a new instance.


In [1]:
from kitefs import FeatureStore
import pandas as pd

In [2]:
store = FeatureStore()

### Optional error-case demos

The notebook still runs the happy path with clean datasets. Error-case cells below are intentionally commented out; uncomment a block when you want to show how KiteFS responds to malformed inputs or invalid parameters.

The convention is to create in-memory copies named `malformed_*`, mutate one thing, and pass that copy to KiteFS. The clean datasets remain unchanged.

### 1.2 Apply & inspect the registry

`apply()` discovers every `FeatureGroup` in `feature_store/definitions/*.py`, validates cross-definition rules (e.g. join key references), and writes `feature_store/registry.json`. The registry is the single source of truth for all downstream operations. Run `apply()` again whenever you change a definition.


In [3]:
result = store.apply()
print(result)

ApplyResult(registered_groups=['demo_filter_features', 'demo_none_features', 'listing_features', 'town_market_features'], published=False)


In [4]:
store.list_feature_groups()

[FeatureGroupSummary(name='demo_filter_features', owner='data-science-team', description='Demo-only group for FILTER validation mode', entity_key='demo_id', storage_target=<StorageTarget.OFFLINE: 'OFFLINE'>, feature_count=1),
 FeatureGroupSummary(name='demo_none_features', owner='data-science-team', description='Demo-only group for NONE validation mode', entity_key='demo_id', storage_target=<StorageTarget.OFFLINE: 'OFFLINE'>, feature_count=1),
 FeatureGroupSummary(name='listing_features', owner='data-science-team', description='Historical sold listing attributes and prices', entity_key='listing_id', storage_target=<StorageTarget.OFFLINE: 'OFFLINE'>, feature_count=4),
 FeatureGroupSummary(name='town_market_features', owner='data-science-team', description='Monthly town-level market aggregate', entity_key='town_id', storage_target=<StorageTarget.OFFLINE_AND_ONLINE: 'OFFLINE_AND_ONLINE'>, feature_count=1)]

**CLI alternative** — `kitefs apply`, `kitefs list`, and `kitefs describe` do the same from the terminal. The cells below run those commands directly in the notebook using `!` shell syntax.


In [5]:
!kitefs list --format json

[
  {
    "name": "demo_filter_features",
    "owner": "data-science-team",
    "description": "Demo-only group for FILTER validation mode",
    "entity_key": "demo_id",
    "storage_target": "OFFLINE",
    "feature_count": 1
  },
  {
    "name": "demo_none_features",
    "owner": "data-science-team",
    "description": "Demo-only group for NONE validation mode",
    "entity_key": "demo_id",
    "storage_target": "OFFLINE",
    "feature_count": 1
  },
  {
    "name": "listing_features",
    "owner": "data-science-team",
    "description": "Historical sold listing attributes and prices",
    "entity_key": "listing_id",
    "storage_target": "OFFLINE",
    "feature_count": 4
  },
  {
    "name": "town_market_features",
    "owner": "data-science-team",
    "description": "Monthly town-level market aggregate",
    "entity_key": "town_id",
    "storage_target": "OFFLINE_AND_ONLINE",
    "feature_count": 1
  }
]


In [6]:
!kitefs describe listing_features --format json

{
  "applied_at": "2026-06-07T09:11:31.300221Z",
  "entity_key": {
    "description": null,
    "dtype": "INTEGER",
    "name": "listing_id"
  },
  "event_timestamp": {
    "description": null,
    "dtype": "DATETIME",
    "name": "sold_at"
  },
  "features": [
    {
      "description": null,
      "dtype": "INTEGER",
      "expect": [
        {
          "type": "not_null"
        },
        {
          "type": "gte",
          "value": 1900
        },
        {
          "type": "lte",
          "value": 2030
        }
      ],
      "name": "build_year"
    },
    {
      "description": null,
      "dtype": "INTEGER",
      "expect": [
        {
          "type": "not_null"
        },
        {
          "type": "gt",
          "value": 0
        }
      ],
      "name": "net_area"
    },
    {
      "description": null,
      "dtype": "INTEGER",
      "expect": [
        {
          "type": "not_null"
        },
        {
          "type": "gt",
          "value": 0
        }
    

### 1.3 Prepare & ingest data

KiteFS does not compute features — your data pipeline does. `ingest()` accepts two input styles: a `pandas.DataFrame` or a file path (`.parquet` or `.csv`). Either way, the data goes through the same schema validation and is written to the offline store as partitioned Parquet under `feature_store/data/offline_store/<group>/year=YYYY/month=MM/`.

In [48]:
listing_features_df = pd.read_parquet("data/listing_features.parquet")

In [9]:
listing_features_df.head()

,listing_id,town_id,net_area,number_of_rooms,build_year,sold_price,sold_at
0,1001,6,48,1,1995,592403.98,2025-02-24 09:43:47+00:00
1,1002,5,97,3,1981,1522575.69,2025-02-08 16:38:01+00:00
2,1003,5,139,3,2014,2359839.54,2025-04-15 17:17:51+00:00
3,1004,1,116,3,1997,3272615.87,2025-03-31 13:06:05+00:00
4,1005,4,117,3,2018,2352991.98,2025-02-02 19:29:34+00:00


In [10]:
# using ingest with dataframe
store.ingest("listing_features", listing_features_df)

# using ingest with file path
result = store.ingest("town_market_features", "data/town_market_features.parquet")
print(result)

IngestResult(feature_group='town_market_features', accepted_rows=72, rejected_rows=0, written_files=['/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=02/ing_20260607T091749_0d0ce8.parquet', '/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=03/ing_20260607T091749_5b3cb4.parquet', '/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=04/ing_20260607T091749_a7f805.parquet', '/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=05/ing_20260607T091749_c7084a.parquet', '/Users/Fedai_Paca/Developer/adu-term-project/kitefs-examples/training/feature_store/data/offline_store/town_market_features/year=2025/month=06/ing_20260607T091749_7ae5a6.parq

**CLI alternative** — `kitefs ingest` accepts file paths only:
```bash
kitefs ingest listing_features data/listing_features.parquet
kitefs ingest town_market_features data/town_market_features.parquet
```


#### Optional error case: ingestion validation

Ingestion always checks that the incoming data matches the registered feature group: required columns must exist, structural columns must be non-null, and column types must align. After that, feature-level `Expect` rules run according to the group's validation mode.

`listing_features` uses `ERROR`, so a schema mismatch or invalid feature value rejects the batch and writes no rows.

In [ ]:
# Wrong feature group name: the registry is the contract.
# Uncomment to see KiteFS list the registered feature groups in the error message.
#
# store.ingest("listings", listing_features_df)

# Schema check: required column does not match the feature group.
# Uncomment to see KiteFS reject the batch before writing anything.
#
# malformed_listing_features_df = listing_features_df.rename(
#     columns={"listing_id": "listing_identifier"}
# )
# store.ingest("listing_features", malformed_listing_features_df)


# Structural validation: column type does not match the feature group.
# Uncomment to see KiteFS report that `net_area` is no longer an INTEGER column.
#
# malformed_listing_features_df = listing_features_df.copy()
# malformed_listing_features_df["net_area"] = malformed_listing_features_df["net_area"].astype(str)
# store.ingest("listing_features", malformed_listing_features_df)


# Value validation: feature values violate the Expect rules in ERROR mode.
# Uncomment to see the whole batch rejected and inspect the validation report.
#
# malformed_listing_features_df = listing_features_df.copy()
# malformed_listing_features_df.loc[0, "net_area"] = 0
# malformed_listing_features_df.loc[1, "build_year"] = 1899
# malformed_listing_features_df.loc[2, "sold_price"] = None
# try:
#     store.ingest("listing_features", malformed_listing_features_df)
# except Exception as err:
#     print(type(err).__name__)
#     print(err)
#     print(err.report)
#     print(err.report.failures[:3])

FeatureGroupNotFoundError: group=listings: feature group not found in registry; registered groups: ['demo_filter_features', 'demo_none_features', 'listing_features', 'town_market_features']. Next: run 'kitefs apply' or pick one of the registered groups

#### Validation modes: ERROR, FILTER, and NONE

The two demo feature groups are intentionally tiny and isolated from the real example data.

- The real groups use `ERROR`: any feature-rule failure rejects the whole ingest batch.
- `demo_filter_features` uses `FILTER`: invalid rows are dropped and valid rows are written.
- `demo_none_features` uses `NONE`: feature-level `Expect` rules are skipped, but structural checks still run.

Because these demo groups live in `feature_store/definitions/`, `store.apply()` registers them along with the real feature groups.

In [18]:
demo_mode_df = pd.DataFrame(
    {
        "demo_id": [1, 2, 3],
        "event_timestamp": pd.to_datetime(
            ["2025-01-01", "2025-01-02", "2025-01-03"], utc=True
        ),
        "score": [10.0, -5.0, 20.0],
    }
)

filter_result = store.ingest("demo_filter_features", demo_mode_df)
print("FILTER accepted rows:", filter_result.accepted_rows)
print("FILTER rejected rows:", filter_result.rejected_rows)
print("FILTER validation report:", filter_result.validation_report)

none_result = store.ingest("demo_none_features", demo_mode_df)
print("NONE accepted rows:", none_result.accepted_rows)
print("NONE rejected rows:", none_result.rejected_rows)
print("NONE validation report:", none_result.validation_report)

FILTER accepted rows: 2
FILTER rejected rows: 1
FILTER validation report: ValidationReport(pass_count=2, fail_count=1, failures=[ValidationFailure(field='score', constraint='gt(0)', actual_value=-5.0, entity_key_value=2, row_index=1)])
NONE accepted rows: 3
NONE rejected rows: 0
NONE validation report: None


In [19]:
! kitefs ingest listing_features data/listing_features.parquet

Ingested 10000 row(s) into 'listing_features'. Rejected 0 row(s). Wrote 12 file(s).
  Validation: 10000 passed, 0 failed.


### 1.4 Explore the offline store

`get_historical_features()` reads directly from the offline Parquet partitions. `select` names the feature columns to return. `where` filters by event timestamp. Structural columns (entity key, event timestamp, join keys) are always included in the result.


In [20]:
from datetime import datetime

listings_wo_join = store.get_historical_features(
    from_="listing_features",
    select=[
        "net_area",
        "number_of_rooms",
        "build_year",
        "sold_price",
    ],
    where={
        "sold_at": {
            "gte": datetime(2025, 4, 1),
            "lte": datetime(2025, 4, 10, 23, 59, 59),
        }
    },
)

listings_wo_join

,listing_id,sold_at,town_id,net_area,number_of_rooms,build_year,sold_price
0,1041,2025-04-09 08:05:40,3,149,3,1996,3330769.37
1,1058,2025-04-10 16:42:40,6,145,3,1990,1974060.03
2,1080,2025-04-04 17:50:02,6,84,2,2023,1130638.85
3,1245,2025-04-02 10:45:21,2,120,3,2012,3483919.20
4,1251,2025-04-01 15:20:26,6,70,1,2024,851586.00
...,...,...,...,...,...,...,...
249,10881,2025-04-02 17:22:07,4,67,2,2021,1559155.95
250,10882,2025-04-06 10:09:28,1,178,4,1985,5614307.75
251,10901,2025-04-06 14:21:56,3,130,3,1998,2796174.51
252,10949,2025-04-03 14:13:39,2,90,2,2005,3127845.62


### Point-in-time join

A **point-in-time join** (also called an *as-of join*) is one of the defining capabilities of a feature store. When you build a training row, you must use only the feature values that were actually known at that row's event timestamp. Pulling in a value computed *later* leaks future information into training and produces a model that looks great offline but underperforms in production.

For each `listing_features` row, KiteFS joins the most recent `town_market_features` row whose `event_timestamp` is **at or before** the listing's `sold_at`. Future market snapshots are never used.

Let's see this on a small subset of the offline data — a single town and one sold listing — before applying it to the full training set in section 1.6.

In [21]:
# Every town_market_features snapshot KiteFS could consider for town_id = 1.
# event_timestamp is stamped on the FIRST day of the following month,
# e.g. the March aggregate only becomes available on 2025-04-01.
market_snapshots = store.get_historical_features(
    from_="town_market_features",
    select=["avg_price_per_sqm"],
    where={
        "event_timestamp": {
            "gte": datetime(2025, 1, 1),
            "lte": datetime(2025, 7, 1),
        }
    },
)

town_1_snapshots = (
    market_snapshots[market_snapshots["town_id"] == 1]
    .sort_values("event_timestamp")
    .reset_index(drop=True)
)
town_1_snapshots

,town_id,event_timestamp,avg_price_per_sqm
0,1,2025-02-01,29963.36
1,1,2025-03-01,29998.14
2,1,2025-04-01,30475.04
3,1,2025-05-01,30390.25
4,1,2025-06-01,30718.13
5,1,2025-07-01,30989.18


Now take a single listing in town 1 that sold in early April 2025. Its `sold_at` is the anchor KiteFS uses to decide which market snapshot is valid for this row.

In [ ]:
# One sold listing in town 1, used as the anchor for the point-in-time join.
# `where` filters only the base group's event timestamp (sold_at), so we narrow
# to town 1 in pandas to keep the example focused on a single row.
one_listing = store.get_historical_features(
    from_="listing_features",
    select=["net_area", "number_of_rooms", "build_year", "sold_price"],
    where={
        "sold_at": {
            "gte": datetime(2025, 4, 1),
            "lte": datetime(2025, 4, 10, 23, 59, 59),
        }
    },
)

one_listing = one_listing[one_listing["town_id"] == 1].head(1).reset_index(drop=True)
one_listing

,listing_id,sold_at,town_id,net_area,number_of_rooms,build_year,sold_price
0,1638,2025-04-07 15:24:43,1,144,3,1986,4403338.16


Now let KiteFS do the join. Adding `join=["town_market_features"]` performs the point-in-time join automatically. The result keeps the listing columns and appends the matched market feature, prefixed with the joined group name: `town_market_features_avg_price_per_sqm`.

In [ ]:
joined = store.get_historical_features(
    from_="listing_features",
    join=["town_market_features"],
    select={
        "listing_features": ["net_area", "number_of_rooms", "build_year", "sold_price"],
        "town_market_features": ["avg_price_per_sqm"],
    },
    where={
        "sold_at": {
            "gte": datetime(2025, 4, 1),
            "lte": datetime(2025, 4, 10, 23, 59, 59),
        }
    },
)

# Same single town-1 listing as above, now with the point-in-time market value attached.
joined = joined[joined["town_id"] == 1].head(1).reset_index(drop=True)
joined

,listing_id,sold_at,town_id,net_area,number_of_rooms,build_year,sold_price,town_market_features_town_id,town_market_features_event_timestamp,town_market_features_avg_price_per_sqm
0,1638,2025-04-07 15:24:43,1,144,3,1986,4403338.16,1,2025-04-01,30475.04


#### What KiteFS did

Our listing sold on an early-April 2025 date. Looking back at the candidate snapshots for town 1:

- The snapshot stamped `2025-04-01` (the March market aggregate) **is** at or before the sale date → this is the value KiteFS attached.
- The snapshot stamped `2025-05-01` (the April aggregate) is **after** the sale date → KiteFS ignored it, even though it already exists in the offline store.

So `town_market_features_avg_price_per_sqm` in the joined row matches the `2025-04-01` snapshot from the candidate table above, not any later one. That is exactly the market information a real system would have had at the moment the listing sold.

By anchoring every training row to its own event timestamp, KiteFS guarantees **point-in-time correctness**: future data can never leak into the past. This is the single most important guarantee a feature store provides for keeping training and serving consistent — and you get it from one `join` argument instead of hand-written, error-prone temporal SQL.

### 1.5 Build a training dataset (point-in-time join) & train the model

We saw the point-in-time join up close in section 1.4. Building a training dataset is the same operation applied across the full history: for every sold listing in the window, KiteFS joins the market snapshot that was current when it sold. The helper below wraps that single call and trains a model on the result.

In [24]:
from datetime import datetime
from pathlib import Path

import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

FEATURE_COLUMNS = [
    "net_area",
    "number_of_rooms",
    "build_year",
    "town_market_features_avg_price_per_sqm",
]
LABEL_COLUMN = "sold_price"


def build_training_dataset(store, *, until):
    return store.get_historical_features(
        from_="listing_features",
        join=["town_market_features"],
        select={
            "listing_features": [
                "net_area",
                "number_of_rooms",
                "build_year",
                "sold_price",
            ],
            "town_market_features": ["avg_price_per_sqm"],
        },
        where={
            "sold_at": {
                "gte": datetime(2025, 2, 1),
                "lte": until,
            }
        },
    )


def train_and_evaluate(training_df):
    X = training_df[FEATURE_COLUMNS]
    y = training_df[LABEL_COLUMN]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    print("MAE:", mean_absolute_error(y_test, predictions))
    print("R2 :", r2_score(y_test, predictions))

    return model


def save_model(model, path="models/model.pkl"):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, path)
    
    print(f"Model saved → {path}")


**Note:** In `build_training_dataset` method we are retrieving historical features starting from February 2025, not January 2025, because the point-in-time join needs to look back at least one month to get the first `town_market_features` record for January 2025. If we started from January 2025, those listings would have no matching market features and would be dropped from the training dataset. This is an architectural decision regarding to model training, not a KiteFS limitation.

In [25]:
training_df = build_training_dataset(store, until=datetime(2025, 12, 31, 23, 59, 59))
training_df.head()


,listing_id,sold_at,town_id,net_area,number_of_rooms,build_year,sold_price,town_market_features_town_id,town_market_features_event_timestamp,town_market_features_avg_price_per_sqm
0,1001,2025-02-24 09:43:47,6,48,1,1995,592403.98,6,2025-02-01,12987.48
1,1002,2025-02-08 16:38:01,5,97,3,1981,1522575.69,5,2025-02-01,17093.62
2,1005,2025-02-02 19:29:34,4,117,3,2018,2352991.98,4,2025-02-01,21044.18
3,1007,2025-02-28 11:55:06,6,47,1,1998,673645.54,6,2025-02-01,12987.48
4,1029,2025-02-25 11:10:26,5,99,2,1980,1850068.46,5,2025-02-01,17093.62


In [26]:
model = train_and_evaluate(training_df)


MAE: 153651.73382779333
R2 : 0.9777948434747418


In [27]:
save_model(model)


Model saved → models/model.pkl


### 1.6 Materialize & read from the online store

Materialization copies, for each entity key, the offline row with the **latest** `event_timestamp` into the online store. In local mode the online store is a SQLite database at `feature_store/data/online_store/online.db`.

Only groups with `StorageTarget.OFFLINE_AND_ONLINE` can be materialized. `listing_features` is offline-only — calling `materialize("listing_features")` would raise `FeatureGroupNotMaterializableError`.


In [28]:
materialize_result = store.materialize("town_market_features")
print(materialize_result)

MaterializeResult(succeeded=['town_market_features'], skipped=[], failed=[])


**CLI alternative** — `kitefs materialize [GROUP_NAME]` does the same from the terminal. Omit the group name to materialize all `OFFLINE_AND_ONLINE` groups at once:
```bash
kitefs materialize town_market_features
kitefs materialize  # all eligible groups
```

In [32]:
!kitefs materialize

Succeeded: town_market_features. Skipped: (none). Failed: (none).


In [ ]:
# `listing_features` is offline-only, so it cannot be materialized to the online store.
# Uncomment to see KiteFS reject the request before doing any online-store work.
#
# store.materialize("listing_features")

FeatureGroupNotMaterializableError: group=listing_features: storage_target is OFFLINE — only OFFLINE_AND_ONLINE groups can be materialized. Next: change the group's storage_target to OFFLINE_AND_ONLINE and re-apply, or omit this group

Since in the local runtime the online store is sqlite, it is possible to connect it with any SQL client. And query on the database directly with SQL syntax.

In [33]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("feature_store/data/online_store/online.db")

df = pd.read_sql_query("select * from town_market_features;", conn)
print(df)

conn.close()

   town_id              event_timestamp  avg_price_per_sqm
0        1  2026-01-01T00:00:00.000000Z           31828.45
1        2  2026-01-01T00:00:00.000000Z           33041.63
2        3  2026-01-01T00:00:00.000000Z           23190.73
3        4  2026-01-01T00:00:00.000000Z           22122.53
4        5  2026-01-01T00:00:00.000000Z           17990.52
5        6  2026-01-01T00:00:00.000000Z           13610.89


Use get_online_features() to read from the online store. It works like get_historical_features with similar syntax, but always returns the latest features for each entity key, since the online store has the most recent data. 

In [34]:
market_price = store.get_online_features(
    from_="town_market_features",
    select=["*"],
    where={"town_id": {"eq": 1}},
)

market_price

{'town_id': 1,
 'event_timestamp': datetime.datetime(2026, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
 'avg_price_per_sqm': 31828.45}

## Phase 2 - Switch to Remote & Publish (AWS)

The local workflow is complete. To share features with the backend service and other team members, switch to the remote target.

**Before running these cells:**
1. Copy `.env.example` → `.env` in this directory.
2. Fill in your S3 bucket name, AWS profile, and region.
3. Confirm your AWS credentials are configured in `~/.aws/config`.

KiteFS uses the standard boto3 credential chain, it does not store or read credentials itself.

### 2.1 Load remote configuration


**Demo definition note:** This showcase includes two tiny validation-mode demo groups, so remote publishing will include them in the registry too. In a production feature store, remove `feature_store/definitions/demo_validation_modes.py` if you only want the real application groups.

In [35]:
import os
from dotenv import load_dotenv

# Load environment variables from .env.
load_dotenv()

print("Runtime target:", os.environ.get("KITEFS_RUNTIME_TARGET", "(not set)"))
print("AWS region    :", os.environ.get("KITEFS_AWS_REGION", "(not set)"))


Runtime target: remote
AWS region    : us-east-1


In [36]:
# Build a fresh, AWS-bound store instance to resolve the environment variables and use them.
remote_store = FeatureStore()

As a side note, running `kitefs apply` will create the instance behind the scenes and load the configuration. So manual instantiation is needed only using SDK. 

### 2.2 Publish the registry and ingest data to S3

`apply(publish=True)` writes the registry to S3 in addition to the local file, making the feature group schemas available to any consumer project (e.g. the backend service). 

KiteFS does not create S3 buckets because of security and unique name requirements. So before moving to this phase, be sure to create the bucket you specified in `.env` and confirm your credentials have write access to it.


In [38]:
apply_result = remote_store.apply(publish=True)
print("Registered groups:", apply_result.registered_groups)
print("Published successfully:", apply_result.published)

Registered groups: ['demo_filter_features', 'demo_none_features', 'listing_features', 'town_market_features']
Published successfully: True


`publish=True` create the registry and uploads it to S3. When using the CLI:
```bash
kitefs apply --publish
```
will prompt you to confirm before proceeding. To bypass the prompt, add `--yes`:
```bash
kitefs apply --publish --yes
```
which is useful for CI/CD pipelines.

In [39]:
print("Ingesting listing features to S3...")
remote_store.ingest("listing_features", "data/listing_features.parquet")

print("Ingesting town market aggregates to S3...")
remote_store.ingest("town_market_features", "data/town_market_features.parquet")

print("Ingestion complete!")

Ingesting listing features to S3...
Ingesting town market aggregates to S3...
Ingestion complete!


In [40]:
remote_store.list_feature_groups()

[FeatureGroupSummary(name='demo_filter_features', owner='data-science-team', description='Demo-only group for FILTER validation mode', entity_key='demo_id', storage_target=<StorageTarget.OFFLINE: 'OFFLINE'>, feature_count=1),
 FeatureGroupSummary(name='demo_none_features', owner='data-science-team', description='Demo-only group for NONE validation mode', entity_key='demo_id', storage_target=<StorageTarget.OFFLINE: 'OFFLINE'>, feature_count=1),
 FeatureGroupSummary(name='listing_features', owner='data-science-team', description='Historical sold listing attributes and prices', entity_key='listing_id', storage_target=<StorageTarget.OFFLINE: 'OFFLINE'>, feature_count=4),
 FeatureGroupSummary(name='town_market_features', owner='data-science-team', description='Monthly town-level market aggregate', entity_key='town_id', storage_target=<StorageTarget.OFFLINE_AND_ONLINE: 'OFFLINE_AND_ONLINE'>, feature_count=1)]

### 2.3 Materialize to the remote online store (DynamoDB)

Materialization now writes to DynamoDB. Materialize will create the DynamoDB table if it doesn't exist, so no manual setup is needed.


In [41]:
materialize_result = remote_store.materialize("town_market_features")
print(materialize_result)


MaterializeResult(succeeded=['town_market_features'], skipped=[], failed=[])


## Phase 3 - Retraining Loop

Assuming one month later, January 2026 data is ready. The workflow is the same as Phase 1. This section uses the **CLI** as the primary surface to demonstrate it as a real alternative to the SDK and how it can be used during CI/CD pipelines. 

### 3.1 Ingest January 2026 data and refresh the online store

The CLI is the primary surface here. Each command is followed by its SDK equivalent in a callout block.


In [42]:
!kitefs ingest listing_features data/listing_features_jan2026.parquet

Ingested 1000 row(s) into 'listing_features'. Rejected 0 row(s). Wrote 1 file(s).
  Validation: 1000 passed, 0 failed.


In [43]:
!kitefs ingest town_market_features data/town_market_jan2026.parquet

Ingested 6 row(s) into 'town_market_features'. Rejected 0 row(s). Wrote 1 file(s).
  Validation: 6 passed, 0 failed.


### 3.2 Rebuild the training dataset and retrain

Now we can move the rolling window forward by one month and retrain the model with the latest data.

In [44]:
training_df = build_training_dataset(
    remote_store,
    until=datetime(2026, 1, 31, 23, 59, 59),
)


In [47]:
training_df.shape

(10144, 10)

In [51]:
training_df.tail()

,listing_id,sold_at,town_id,net_area,number_of_rooms,build_year,sold_price,town_market_features_town_id,town_market_features_event_timestamp,town_market_features_avg_price_per_sqm
10139,20996,2026-01-30 18:57:05,4,115,3,1983,2545157.35,4,2026-01-01,22122.53
10140,20997,2026-01-24 10:03:56,2,128,3,1999,4166174.46,2,2026-01-01,33041.63
10141,20998,2026-01-04 13:41:45,2,218,5,2002,6757360.75,2,2026-01-01,33041.63
10142,20999,2026-01-10 17:14:14,1,176,4,1996,5401933.36,1,2026-01-01,31828.45
10143,21000,2026-01-22 12:50:35,6,187,4,2013,2371195.20,6,2026-01-01,13610.89


In [52]:
model = train_and_evaluate(training_df)
save_model(model)


MAE: 151370.2854069356
R2 : 0.9785126010713899
Model saved → models/model.pkl


### 3.3 Materialize the latest features and read from the online store

In [53]:
!kitefs materialize town_market_features

Succeeded: town_market_features. Skipped: (none). Failed: (none).


In [54]:
market_price = remote_store.get_online_features(
    from_="town_market_features",
    select=["*"],
    where={"town_id": {"eq": 1}},
)

print(market_price)

{'town_id': 1, 'event_timestamp': datetime.datetime(2026, 2, 1, 0, 0, tzinfo=datetime.timezone.utc), 'avg_price_per_sqm': 31346.96}
